# 03 — VAE: detección de anomalías viales

Este notebook **no entrena**: lee los resultados generados por

```powershell
python -m src.data.cache_images --size 128
python -m src.data.cache_images --size 512
python -m src.models.train_vae --config configs/vae/vae_base.yaml
python -m src.models.train_vae --config configs/vae/vae_patch512.yaml
```

| Corrida | Qué ve el VAE |
|---|---|
| `vae_base` | la imagen completa a 128×128 |
| `vae_patch512` | la imagen a 512×512 cortada en 64 parches de 64×64 (cada parche por separado) |

Ambas se entrenan solo con imágenes de `train` **sin daño anotado** (`has_any_damage == 0`) y eligen el checkpoint
con la pérdida (β = 1) en las imágenes sin daño de `val`, sin usar etiquetas. Cada una produce tres scores:

- **global**: error cuadrático medio de toda la imagen;
- **patch**: máximo del error promediado por celda (16×16 en `vae_base`, 64×64 en `vae_patch512`);
- **patch_norm**: máximo de `(error de la celda − media de esa posición) / desviación de esa posición`, con
  media y desviación calculadas solo en imágenes de `train` sin daño. Resta lo que "siempre" cuesta reconstruir
  en cada posición (cielo, postes, capó).

Para cada score el threshold maximiza F1 de D40 en `val`; el score principal es el de mayor PR-AUC en `val`.
Todo queda fijo para `test_id`, `test_ood` (Norway) y `test_perspective` (China_Drone).
Las corridas `_limitN` son pruebas y se ignoran.

**Criterio de éxito (fijado antes de ver resultados):** PR-AUC de D40 en `val` claramente por encima de la
prevalencia y AUROC `no_damage_vs_pothole` > 0.6 **dentro** de los países que tienen ambos grupos.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS = PROJECT_ROOT / "reports" / "experiments" / "vae"
SPLITS = ["val", "test_id", "test_ood", "test_perspective"]
GROUPS = ["no_damage", "other", "pothole"]

runs = {}
for folder in sorted(RESULTS.glob("*")) if RESULTS.exists() else []:
    if (folder / "metrics.json").exists() and "_limit" not in folder.name:
        runs[folder.name] = dict(
            config=yaml.safe_load((folder / "config.yaml").read_text(encoding="utf-8")),
            metrics=json.loads((folder / "metrics.json").read_text(encoding="utf-8")),
            history=pd.read_csv(folder / "history.csv"),
            predictions=pd.read_csv(folder / "predictions.csv"),
        )
print("Corridas encontradas:", list(runs) or "ninguna todavía")

## 1. Configuración registrada

In [ ]:
keys = ["latent", "image_size", "patch_size", "patch", "batch_size", "epochs", "lr", "patience", "warmup_epochs",
        "std_floor_frac", "optimizer", "best_epoch", "train_images", "val_no_damage_images", "image_source", "seed",
        "dataset_version", "splits_sha256"]
pd.DataFrame({name: {k: run["config"].get(k) for k in keys} for name, run in runs.items()})

## 2. Curvas de entrenamiento (una figura por corrida)

La pérdida de `val` es solo sobre imágenes sin daño y siempre con β = 1; mientras β < 1 (warmup) no se elige
checkpoint. Las pérdidas absolutas **no** son comparables entre corridas (cada imagen suma 49 mil píxeles en
`vae_base` y 786 mil en `vae_patch512`). La PR-AUC de `val` se registra por época pero no interviene en la selección.

In [ ]:
for name, run in runs.items():
    h = run["history"]
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))
    axes[0].plot(h.epoch, h.train_loss, label="train")
    axes[0].plot(h.epoch, h.val_loss, "--", label="val (sin daño, β=1)")
    axes[1].plot(h.epoch, h.train_recon, label="train")
    axes[1].plot(h.epoch, h.val_recon, "--", label="val")
    axes[2].plot(h.epoch, h.train_kl, label="train")
    axes[2].plot(h.epoch, h.val_kl, "--", label="val")
    axes[2].plot(h.epoch, h.beta * h.train_kl.max(), ":", color="gray", label="β (escalado)")
    axes[3].plot(h.epoch, h.val_pr_auc_global, label="global")
    axes[3].plot(h.epoch, h.val_pr_auc_patch, label="patch")
    axes[3].axvline(run["metrics"]["best_epoch"], color="k", lw=0.8, label="checkpoint")
    for ax, title in zip(axes, ["Pérdida total", "Reconstrucción", "KL", "PR-AUC D40 en val"]):
        ax.set(title=title, xlabel="epoch")
        ax.legend(fontsize=8)
    fig.suptitle(name)
    plt.tight_layout()

## 3. Métricas D40: corridas × scores

La PR-AUC se lee junto a la prevalencia (PR-AUC de un clasificador aleatorio). `f1_trivial` = 2p / (1 + p) es el
F1 de marcar todo como D40: un F1 que no lo supera no aporta nada.

In [ ]:
rows = []
for name, run in runs.items():
    m = run["metrics"]
    print(f"{name}: score principal = {m['primary_score']}, PR-AUC val = "
          + ", ".join(f"{k} {v:.4f}" for k, v in m["val_pr_auc"].items()))
    for split, by_score in m["splits"].items():
        for score, values in by_score.items():
            p = values["prevalence"]
            rows.append(dict(experiment=name, score=score, split=split, prevalence=p, pr_auc=values["pr_auc"],
                             f1=values["f1"], f1_trivial=2 * p / (1 + p), recall=values["recall"],
                             precision=values["precision"]))
table = pd.DataFrame(rows)
table.pivot(index=["experiment", "score"], columns="split", values=["pr_auc", "f1"]).reindex(
    columns=SPLITS, level=1).round(4) if not table.empty else "Sin resultados todavía"

In [ ]:
if not table.empty:
    display(table.groupby("split")[["prevalence", "f1_trivial"]].first().reindex(SPLITS).round(4))
    order = ["val", "test_id", "test_ood"]
    fig, axes = plt.subplots(1, len(order), figsize=(5 * len(order), 3.6), sharey=False)
    for ax, split in zip(axes, order):
        part = table[table.split == split].pivot(index="score", columns="experiment", values="pr_auc")
        part.plot.bar(ax=ax, rot=0)
        ax.axhline(table[table.split == split].prevalence.iloc[0], color="k", ls="--", lw=0.8, label="prevalencia")
        ax.set(title=split, ylabel="PR-AUC D40")
        ax.legend(fontsize=7)
    plt.tight_layout()

## 4. ¿Detecta baches o cualquier daño?

Score principal de cada corrida. `flagged_rate` es la proporción sobre el threshold: en `no_damage` y `other` son
falsos positivos, en `pothole` es el recall. AUROC por pares: 0.5 = no separa; > 0.5 = el segundo grupo puntúa más.

In [ ]:
for name, run in runs.items():
    primary = run["metrics"]["primary_score"]
    print(f"{name} ({primary})")
    rows = [dict(split=split, group=group, **values)
            for split, report in run["metrics"]["breakdown"].items()
            for group, values in report[primary]["groups"].items()]
    display(pd.DataFrame(rows).pivot(index="group", columns="split", values=["n", "median", "flagged_rate"])
            .reindex(columns=SPLITS, level=1).reindex(GROUPS).round(4))
    rows = [dict(split=split, score=score, pair=pair, auroc=values["auroc"])
            for split, report in run["metrics"]["breakdown"].items()
            for score, by_score in report.items()
            for pair, values in by_score["pairwise_auroc"].items()]
    display(pd.DataFrame(rows).pivot(index=["score", "pair"], columns="split", values="auroc")
            .reindex(columns=SPLITS).round(3))

In [ ]:
for name, run in runs.items():
    predictions = run["predictions"]
    fig, axes = plt.subplots(1, len(SPLITS), figsize=(4.5 * len(SPLITS), 3.4))
    for ax, split in zip(axes, SPLITS):
        part = predictions[predictions.split == split]
        for group in GROUPS:
            values = part.loc[part.group == group, "y_score"]
            if len(values):
                ax.hist(values, bins=50, density=True, histtype="step", label=f"{group} (n={len(values)})")
        ax.axvline(part.threshold.iloc[0], color="k", lw=0.8)
        ax.set(title=split, xlabel=f"score ({run['metrics']['primary_score']})")
        ax.legend(fontsize=7)
    fig.suptitle(f"{name}: distribución del score principal por grupo (línea = threshold de val)")
    plt.tight_layout()

## 5. Comparación dentro de cada país

Las imágenes sin daño de `train` y `val` vienen solo de Czech, India y Japan; United_States y China_MotorBike no
tienen ninguna. Un score alto en esos países puede deberse al dominio y no al daño, así que la comparación limpia
es dentro de cada país que tiene los dos grupos. Esta tabla es la que decide el criterio de éxito.

In [ ]:
rows = []
for name, run in runs.items():
    for split in ["val", "test_id"]:
        for score, report in run["metrics"]["breakdown"][split].items():
            for country, values in report["by_country"].items():
                if values["n"]["no_damage"] and values["n"]["pothole"]:
                    rows.append(dict(experiment=name, score=score, split=split, country=country,
                                     n_pothole=values["n"]["pothole"], pr_auc_D40=values["pr_auc_D40"],
                                     auroc_no_damage_vs_pothole=values["pairwise_auroc"]["no_damage_vs_pothole"]["auroc"],
                                     auroc_other_vs_pothole=values["pairwise_auroc"]["other_vs_pothole"]["auroc"]))
within = pd.DataFrame(rows)
within.pivot(index=["experiment", "score"], columns=["split", "country"],
             values="auroc_no_damage_vs_pothole").round(3) if not within.empty else "Sin resultados todavía"

## 6. ¿Con qué otros daños se confunde?

Imágenes **sin D40**, separadas por cada clase anotada (score principal). Una imagen con varias clases cuenta en
cada una. `flagged_rate` = falsos positivos con el threshold de `val`.

In [ ]:
for name, run in runs.items():
    primary = run["metrics"]["primary_score"]
    print(f"{name} ({primary})")
    rows = [dict(split=split, damage=damage, **values)
            for split, report in run["metrics"]["breakdown"].items()
            for damage, values in report[primary]["other_damage_classes"].items()]
    display(pd.DataFrame(rows).pivot(index="damage", columns="split", values=["n", "flagged_rate"])
            .reindex(columns=SPLITS, level=1).round(4))

## 7. Error típico por posición (`patch_norm`)

Media y desviación del error por celda en imágenes de `train` sin daño, con el checkpoint elegido. Celdas con
media alta son las que el VAE siempre reconstruye mal; `patch_norm` las descuenta.

In [ ]:
for name, run in runs.items():
    stats = run["metrics"].get("position_stats")
    if not stats:
        continue
    rows, cols = stats["grid"]
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
    for ax, key in zip(axes, ["mean", "std_eff"]):
        image = ax.imshow(np.array(stats[key]).reshape(rows, cols), cmap="viridis")
        fig.colorbar(image, ax=ax, fraction=0.046)
        ax.set(title=key, xticks=[], yticks=[])
    fig.suptitle(f"{name}: grid {rows}×{cols}, {stats['n_images']} imágenes, {stats['n_floored']} celdas con piso")
    plt.tight_layout()

## 8. Reconstrucciones y mapas de error

Ejemplos de `val` con el checkpoint elegido: original, reconstrucción y error por píxel. Se muestran las dos
imágenes con mayor score principal de cada grupo; en las de D40 se dibujan las cajas de `annotations.csv`
escaladas al tamaño de entrada. El mapa de error no es una segmentación.

In [ ]:
import sys

import torch
from matplotlib.patches import Rectangle

sys.path.insert(0, str(PROJECT_ROOT))  # para importar src desde notebooks/
from src.data.cache_images import cache_root, cached_frame
from src.data.dataset import DATASET_ROOT, PotholeDataset, build_transforms, load_split_frame
from src.models.vae import build_vae, error_map

annotations = pd.read_csv(PROJECT_ROOT / "data" / "manifests" / "annotations.csv")
frame = load_split_frame(["val"]).set_index("sample_id")
for name, run in runs.items():
    config, predictions = run["config"], run["predictions"]
    checkpoint = PROJECT_ROOT / "checkpoints" / "vae" / name / "best.pt"
    if not checkpoint.exists():
        print(f"{name}: falta {checkpoint.relative_to(PROJECT_ROOT)} (los checkpoints no van a Git)")
        continue
    size = config["image_size"]
    model = build_vae(config["latent"], size, config.get("patch_size"))
    model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True)["model"])
    model.eval()
    val = predictions[predictions.split == "val"]
    picks = val.sort_values("y_score", ascending=False).groupby("group").head(2).sort_values("group")
    rows = frame.loc[picks.sample_id].reset_index()
    root = cache_root(size) if config["use_cache"] else DATASET_ROOT
    images = cached_frame(rows) if config["use_cache"] else rows
    dataset = PotholeDataset(images, build_transforms(size=size, normalize=False), root)
    fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
    for i, (_, row) in enumerate(rows.iterrows()):
        image = dataset[i][0].unsqueeze(0)
        with torch.no_grad():
            recon = model(image)[0]
        errors = error_map(image, recon)[0, 0]
        score = picks.set_index("sample_id").loc[row.sample_id]
        axes[i, 0].imshow(image[0].permute(1, 2, 0))
        axes[i, 1].imshow(recon[0].permute(1, 2, 0).clamp(0, 1))
        axes[i, 2].imshow(errors, cmap="inferno")
        scale_x, scale_y = size / row.width, size / row.height
        for box in annotations[(annotations.sample_id == row.sample_id) & (annotations["class"] == "D40")].itertuples():
            for ax in (axes[i, 0], axes[i, 2]):
                ax.add_patch(Rectangle((box.xmin * scale_x, box.ymin * scale_y), box.bbox_width * scale_x,
                                       box.bbox_height * scale_y, fill=False, edgecolor="cyan", lw=1.2))
        axes[i, 0].set_title(f"{score.group} · {row.country}", fontsize=8)
        axes[i, 2].set_title(f"score {score.y_score:.4f}", fontsize=8)
        for ax in axes[i]:
            ax.axis("off")
    fig.suptitle(f"{name}: original · reconstrucción · error")
    plt.tight_layout()

## 9. Latencia (batch 1, solo forward con z = μ; no incluye el cálculo del score)

In [ ]:
pd.DataFrame({name: run["metrics"]["latency"] for name, run in runs.items()}).T

## 10. Comparación con ResNet

Misma métrica (PR-AUC de D40) sobre los mismos splits. Solo aparece si ya existen resultados de ResNet.

In [ ]:
resnet_rows = []
for folder in sorted((PROJECT_ROOT / "reports" / "experiments" / "resnet").glob("*")):
    if (folder / "metrics.json").exists() and "_limit" not in folder.name:
        metrics = json.loads((folder / "metrics.json").read_text(encoding="utf-8"))
        resnet_rows += [dict(model=f"resnet · {folder.name}", split=split, pr_auc=values["pr_auc"], f1=values["f1"])
                        for split, values in metrics["splits"].items()]
vae_rows = [dict(model=f"vae · {name} ({run['metrics']['primary_score']})", split=split,
                 pr_auc=by_score[run["metrics"]["primary_score"]]["pr_auc"],
                 f1=by_score[run["metrics"]["primary_score"]]["f1"])
            for name, run in runs.items() for split, by_score in run["metrics"]["splits"].items()]
if resnet_rows:
    display(pd.DataFrame(resnet_rows + vae_rows).pivot(index="model", columns="split", values=["pr_auc", "f1"])
            .reindex(columns=SPLITS, level=1).round(4))
else:
    print("Todavía no hay resultados de ResNet en reports/experiments/resnet")

## 11. Limitaciones

- **Confounding por país:** United_States y China_MotorBike no tienen imágenes sin daño; el VAE nunca vio su
  aspecto normal y las estadísticas de `patch_norm` solo reflejan las cámaras de Czech, India y Japan (sección 5).
- **Resolución:** un D40 típico de Norway mide ~3 px a 128 y ~11 px a 512; dentro de una celda de 64×64 ocupa ~3 %.
- **`test_perspective`:** solo 5 imágenes sin daño; sus comparaciones con ese grupo no son concluyentes.
- **"Sin daño anotado" ≠ carretera perfecta:** los XML vacíos pueden tener daño no anotado.
- **Threshold:** cuando un score no separa D40, el F1 máximo de `val` degenera en marcar casi todo (ver `f1_trivial`).
- **Reproducibilidad:** la evaluación es determinista (z = μ), pero el entrenamiento en GPU no es idéntico bit a bit.